# 1. Nichts-Modell (Mehrheitsklassen-Baseline)

Klassifikator, der bei jedem Artikel stur dieselbe Kategorie vorhersagt — ohne den Text anzusehen.

- Mehrheitsklasse ausschließlich aus `train.csv` (Split-Regel aus 2.9 / 2i)
- Trefferquote getrennt für Training und Test

In [2]:
import pandas as pd
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "data" / "train.csv").exists() else Path.cwd().parent

train = pd.read_csv(
    root / "data" / "train.csv",
    sep=";",
    quotechar="'",
    header=None,
    names=["label", "text"],
    encoding="utf-8",
)
test = pd.read_csv(
    root / "data" / "test.csv",
    sep=";",
    quotechar="'",
    header=None,
    names=["label", "text"],
    encoding="utf-8",
)

# Mehrheitsklasse ausschließlich aus train.csv (Split-Regel)
mehrheit = train["label"].value_counts().idxmax()

# Nichts-Modell: für jeden Artikel dieselbe Vorhersage
train_pred = pd.Series(mehrheit, index=train.index)
test_pred = pd.Series(mehrheit, index=test.index)

# Trefferquote = Anteil korrekt vorhergesagter Artikel
acc_train = (train_pred == train["label"]).mean()
acc_test = (test_pred == test["label"]).mean()

n_train_korrekt = int((train_pred == train["label"]).sum())
n_test_korrekt = int((test_pred == test["label"]).sum())

print(f"Mehrheitsklasse (aus train.csv): {mehrheit}")
print(f"  Anteil in train: {train['label'].value_counts()[mehrheit]} / {len(train)}")
print()
print(f"Trefferquote train: {acc_train:.4f} ({n_train_korrekt} / {len(train)} = {acc_train:.1%})")
print(f"Trefferquote test:  {acc_test:.4f} ({n_test_korrekt} / {len(test)} = {acc_test:.1%})")

Mehrheitsklasse (aus train.csv): Panorama
  Anteil in train: 1510 / 9245

Trefferquote train: 0.1633 (1510 / 9245 = 16.3%)
Trefferquote test:  0.1634 (168 / 1028 = 16.3%)


## 2. Vergleich der Trefferquoten (zu Schritt 3.1)

| Menge | Trefferquote |
|-------|--------------|
| `train.csv` | ≈ **16,3 %** |
| `test.csv`  | ≈ **16,3 %** |

**Warum sind die Werte fast gleich?**

Das Nichts-Modell rät immer „Panorama“. Es liegt also genau dann richtig, wenn der Artikel wirklich Panorama ist.

In `train.csv` und `test.csv` ist der Anteil an Panorama-Artikeln fast derselbe (je ca. 16 %). Deshalb liegt das Modell auf beiden Mengen etwa gleich oft richtig.

Das kommt daher, dass die Daten bewusst so aufgeteilt wurden, dass beide Dateien dieselbe Klassenverteilung haben.

## 3. Weitere Kennzahlen für die Güte (Recherche)

Neben der **Trefferquote** (Accuracy) gibt es weitere Kennzahlen:

| Kennzahl | Einfach gesagt |
|----------|----------------|
| **Präzision** (Precision) | Von allen Artikeln, die das Modell als z. B. „Sport“ vorhersagt: Wie viele waren wirklich Sport? |
| **Recall** (Sensitivität) | Von allen Artikeln, die wirklich z. B. Sport sind: Wie viele hat das Modell erkannt? |
| **F1-Maß** | Mittelwert aus Präzision und Recall — eine Zahl, die beides zusammenfasst |
| **Konfusionsmatrix** | Tabelle, die zeigt, welche Kategorien wie oft verwechselt wurden |

**Warum das wichtig ist:** Die Trefferquote allein kann täuschen. Bei ungleich großen Klassen (bei uns z. B. Panorama viel häufiger als Kultur) kann ein Modell gut wirken, obwohl es kleine Kategorien kaum trifft. Präzision, Recall und F1 zeigen das besser.

## 4. Welche Kennzahlen entlarven das Nichts-Modell?

| Kennzahl | Belohnt oder entlarvt? | Warum? |
|----------|------------------------|--------|
| **Trefferquote** | eher belohnt | Liegt bei uns schon bei ~16 % — nur weil Panorama so häufig vorkommt. Das Modell „kann“ nichts, wirkt aber nicht bei null. |
| **Präzision** (je Klasse) | gemischt | Für Panorama wirkt sie okay (viele Vorhersagen treffen zufällig). Für alle anderen Klassen gibt es **keine** Vorhersage → Präzision 0 oder undefiniert. |
| **Recall** (je Klasse) | **entlarvt** | Nur Panorama hat Recall 1 (alles wird als Panorama geraten). Alle anderen Klassen haben Recall **0** — nichts wurde erkannt. |
| **F1-Maß** (besonders macro) | **entlarvt** | F1 pro Klasse: Panorama mittelmäßig, alle anderen **0**. Der Mittelwert über alle Klassen ist deshalb sehr niedrig. |
| **Konfusionsmatrix** | **entlarvt** | Man sieht sofort: eine Spalte ist voll (immer Panorama), der Rest ist leer. |

**Fazit:** Die Trefferquote allein belohnt das Nichts-Modell mit dem Anteil der Mehrheitsklasse. **Recall, F1 (macro) und die Konfusionsmatrix** zeigen klar, dass kleine Kategorien komplett ignoriert werden — und entlarven das Modell damit.

## 5. Unser Metrik-Set

Ab jetzt verwenden wir gemeinsam:

1. **Trefferquote** (Accuracy) — grober Überblick
2. **Recall je Klasse** + **macro-Recall** — zeigt, welche Kategorien überhaupt erkannt werden
3. **F1-Maß je Klasse** + **macro-F1** — fasst Präzision und Recall zusammen und bestraft einseitige Modelle

**Warum genau diese?** Die Trefferquote allein belohnt das Nichts-Modell. Recall und F1 (macro) entlarven es: Kategorien, die nie vorhergesagt werden, landen bei 0. So sehen wir später klar, ob ein echtes Modell besser ist als „immer Panorama“.

In [3]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)

labels = sorted(train["label"].unique())


def metrik_set(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    recall_macro = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"=== {name} ===")
    print(f"Trefferquote:   {acc:.4f} ({acc:.1%})")
    print(f"macro-Recall:   {recall_macro:.4f}")
    print(f"macro-F1:       {f1_macro:.4f}")
    print()
    print(classification_report(y_true, y_pred, labels=labels, digits=3, zero_division=0))
    print("Konfusionsmatrix (Zeilen = wahr, Spalten = vorhergesagt):")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(pd.DataFrame(cm, index=labels, columns=labels))
    print()


metrik_set(train["label"], train_pred, "train.csv")
metrik_set(test["label"], test_pred, "test.csv")

=== train.csv ===
Trefferquote:   0.1633 (16.3%)
macro-Recall:   0.1111
macro-F1:       0.0312

               precision    recall  f1-score   support

         Etat      0.000     0.000     0.000       601
       Inland      0.000     0.000     0.000       913
International      0.000     0.000     0.000      1360
       Kultur      0.000     0.000     0.000       485
     Panorama      0.163     1.000     0.281      1510
        Sport      0.000     0.000     0.000      1081
          Web      0.000     0.000     0.000      1509
   Wirtschaft      0.000     0.000     0.000      1270
 Wissenschaft      0.000     0.000     0.000       516

     accuracy                          0.163      9245
    macro avg      0.018     0.111     0.031      9245
 weighted avg      0.027     0.163     0.046      9245

Konfusionsmatrix (Zeilen = wahr, Spalten = vorhergesagt):
               Etat  Inland  International  Kultur  Panorama  Sport  Web  \
Etat              0       0              0       0  

### Ergebnis: Woran man das Nichts-Modell erkennt

| Kennzahl | train | test |
|----------|------:|-----:|
| Trefferquote | 16,3 % | 16,3 % |
| macro-Recall | 0,11 | 0,11 |
| macro-F1 | 0,03 | 0,03 |

**Konkret sichtbar:**

- **8 von 9 Kategorien** haben Recall = 0 und F1 = 0 (Etat, Inland, International, Kultur, Sport, Web, Wirtschaft, Wissenschaft). Das Modell sagt sie **nie** voraus — und trifft sie deshalb nie.
- Nur **Panorama** hat Recall = 1,0 (weil *alles* als Panorama geraten wird) und F1 ≈ 0,28.
- Die **Konfusionsmatrix** hat nur in der Spalte „Panorama“ Einträge; alle anderen Spalten sind 0.

Die Trefferquote (~16 %) wirkt noch „irgendwie okay“. **macro-Recall (~0,11)** und **macro-F1 (~0,03)** zeigen aber: Es wurde nichts gelernt — nur die häufigste Klasse blind ausgegeben.

**Begründung der Wahl:** Genau diese Kennzahlen belohnen kein „immer dieselbe Klasse“-Raten, sondern verlangen, dass *jede* Kategorie erkannt wird. Damit haben wir eine ehrliche Baseline für spätere echte Modelle.

## 6. Was steht fest — und was fehlt noch?

### Fest steht

| Baustein | Unser Stand |
|----------|-------------|
| **Werkbank** | Python, pandas, Jupyter — Daten laden und auswerten funktioniert |
| **Datengrundlage** | 10kGNAD: `train.csv` (~9 245 Artikel) und `test.csv` (~1 028 Artikel), 9 Kategorien, Lizenz geprüft |
| **Gütemaß** | Metrik-Set: Trefferquote, Recall (je Klasse + macro), F1 (je Klasse + macro); Baseline vom Nichts-Modell bekannt |
| **Split-Regel** | Entwickeln und entscheiden nur mit `train.csv`; `test.csv` erst am Ende einmalig prüfen |

### Was noch fehlt

Ein **tatsächlich lernendes System**: etwas, das den **Text** anschaut, daraus Muster ableitet und für *verschiedene* Kategorien Vorhersagen trifft — nicht nur „immer Panorama“.

### Offene Frage für den nächsten Lernjob

**Wie bringen wir dem Computer bei, aus dem Artikeltext zu lernen, welche Kategorie gemeint ist — und wie prüfen wir fair, ob er besser ist als unser Nichts-Modell?**